In [ ]:
!pip install numpy pandas scikit-learn

# RESUME PARSER

In [ ]:
# ==========================================
# CELL 1: Environment Setup & Automated Dependencies
# ==========================================

import sys
import subprocess

REQUIRED_PACKAGES = [
    "pdfplumber",
    "pymupdf",
    "spacy",
    "pandas",
    "numpy",
    "scikit-learn"
]

def install_and_import():
    """Ensures required dependencies are installed without manual pip commands."""
    for pkg in REQUIRED_PACKAGES:
        try:
            __import__(pkg if pkg != "pymupdf" else "fitz")
        except ImportError:
            print(f"Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

install_and_import()

# Standard & Third-Party Imports
import re
import json
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Any, Union, Optional

import fitz  # PyMuPDF
import pdfplumber
import spacy
import pandas as pd
import numpy as np

# Load spaCy NLP Model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model 'en_core_web_sm'...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# print("✓ Cell 1 Complete: All dependencies and NLP models loaded successfully.")
# ==========================================
# CELL 2: Master ResumeParser Engine & File Upload Handler
# ==========================================

from google.colab import files

class ResumeParser:
    """
    Production-grade Resume Parser V2 Engine.
    Handles multi-column layout sorting, alias-based section segmentation,
    and specialized entity extraction into structured JSON outputs.
    """

    KNOWN_HEADERS = {
        "ACADEMIC HISTORY": "EDUCATION",
        "EDUCATION": "EDUCATION",
        "EDUCATIONAL QUALIFICATION": "EDUCATION",
        "TECHNICAL SKILLS": "SKILLS",
        "SKILLS": "SKILLS",
        "EXPERIENCE & PROJECTS": "COMBINED_EXP_PROJ",
        "WORK EXPERIENCE": "EXPERIENCE",
        "EXPERIENCE": "EXPERIENCE",
        "PROJECTS": "PROJECTS",
        "CO-CURRICULAR ACTIVITY": "CERTIFICATIONS",
        "CERTIFICATIONS": "CERTIFICATIONS",
        "EXTRA-CURRICULAR ACTIVITY": "EXTRA_CURRICULAR",
        "LANGUAGE PREFERENCES": "LANGUAGES",
        "LANGUAGES": "LANGUAGES",
        "CAREER OBJECTIVE": "OBJECTIVE",
        "OBJECTIVE": "OBJECTIVE"
    }

    SKILL_TAXONOMY = {
        "programming_languages": [
            r"Python", r"Java", r"C", r"C\+\+", r"JavaScript", r"TypeScript", r"Go", r"Rust", r"PHP", r"Ruby", r"SQL"
        ],
        "frameworks_and_libraries": [
            r"React", r"Angular", r"Vue", r"Node\.js", r"Express", r"Django", r"Flask", r"FastAPI",
            r"Pandas", r"NumPy", r"Scikit-learn", r"TensorFlow", r"PyTorch", r"Keras", r"Tailwind", r"MERN"
        ],
        "databases": [
            r"MySQL", r"PostgreSQL", r"MongoDB", r"SQLite", r"Redis", r"Oracle"
        ],
        "tools_and_cloud": [
            r"Git", r"GitHub", r"Docker", r"Kubernetes", r"AWS", r"Azure", r"GCP", r"Linux",
            r"Power BI", r"Arduino", r"Microcontroller"
        ],
        "soft_skills": [
            r"Problem Solving", r"Leadership", r"Communication", r"Teamwork",
            r"Time Management", r"Adaptability", r"User Testing", r"User Research"
        ]
    }

    DEGREE_PATTERNS = ["bachelor", "master", "b.tech", "higher secondary", "secondary", "class xii", "class x"]

    def __init__(self, pdf_source: Union[str, bytes]):
        self.pdf_source = pdf_source
        self.raw_text: str = ""
        self.sections: Dict[str, List[str]] = {}

    def extract_layout_sorted_text(self) -> str:
        """Extracts text blocks sorted top-to-bottom, left-to-right to support 2-column layouts."""
        if isinstance(self.pdf_source, str):
            doc = fitz.open(self.pdf_source)
        else:
            doc = fitz.open(stream=self.pdf_source, filetype="pdf")

        full_text = ""
        for page in doc:
            blocks = page.get_text("blocks")
            sorted_blocks = sorted(blocks, key=lambda b: (round(b[1] / 10) * 10, b[0]))
            for b in sorted_blocks:
                text = b[4].strip()
                if text:
                    full_text += text + "\n"

        doc.close()
        # Sanitize smart quotes globally
        self.raw_text = full_text.replace('“', '"').replace('”', '"').replace('’', "'")
        return self.raw_text

    def extract_sections(self) -> Dict[str, List[str]]:
        """Segments raw text into canonical sections while eliminating section label artifacts."""
        if not self.raw_text:
            self.extract_layout_sorted_text()

        sections = {
            "HEADER": [], "OBJECTIVE": [], "EDUCATION": [], "SKILLS": [],
            "EXPERIENCE": [], "PROJECTS": [], "CERTIFICATIONS": [],
            "EXTRA_CURRICULAR": [], "LANGUAGES": []
        }

        current_sec = "HEADER"
        SUB_HEADER_LABELS = {"CERTIFICATIONS", "CERTIFICATION", "RESEARCH PAPER", "ACTIVITIES", "ACTIVITY"}

        for line in self.raw_text.splitlines():
            clean_line = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\xff]', '', line).strip()
            if not clean_line:
                continue

            clean_line_no_bullet = re.sub(r'^[•\-\*\x95]\s*', '', clean_line)
            normalized_header = re.sub(r'[\s:]+$', '', clean_line_no_bullet.upper())

            if normalized_header in SUB_HEADER_LABELS:
                if normalized_header in self.KNOWN_HEADERS:
                    current_sec = self.KNOWN_HEADERS[normalized_header]
                continue

            if normalized_header in self.KNOWN_HEADERS:
                current_sec = self.KNOWN_HEADERS[normalized_header]
                continue

            if current_sec == "COMBINED_EXP_PROJ":
                sections["EXPERIENCE"].append(clean_line_no_bullet)
                sections["PROJECTS"].append(clean_line_no_bullet)
            else:
                sections[current_sec].append(clean_line_no_bullet)

        self.sections = sections
        return sections

    def extract_personal_information(self) -> Dict[str, Optional[str]]:
        """Extracts candidate contact info and name."""
        header_lines = self.sections.get("HEADER", [])

        email_match = re.search(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', self.raw_text)
        phone_match = re.search(r'(\+?\d{1,3}[\s-]?)?\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{4}', self.raw_text)
        linkedin_match = re.search(r'(https?://)?(www\.)?linkedin\.com/in/[A-Za-z0-9_-]+/?', self.raw_text, re.IGNORECASE)
        github_match = re.search(r'(https?://)?(www\.)?github\.com/[A-Za-z0-9_-]+/?', self.raw_text, re.IGNORECASE)

        candidate_name = None
        ignore_words = {"resume", "curriculum", "cv", "header", "github", "linkedin"}
        for line in header_lines[:5]:
            clean = line.strip()
            if not clean or "@" in clean or "http" in clean or re.search(r'\d', clean):
                continue
            if any(ig in clean.lower() for ig in ignore_words):
                continue
            words = clean.split()
            if 2 <= len(words) <= 4:
                clean_name = re.sub(r'[^A-Za-z\s]', '', clean).strip()
                if clean_name:
                    candidate_name = clean_name.title()
                    break

        return {
            "name": candidate_name,
            "email": email_match.group().strip() if email_match else None,
            "phone": phone_match.group().strip() if phone_match else None,
            "linkedin": (linkedin_match.group() if linkedin_match.group().startswith("http") else "https://" + linkedin_match.group()) if linkedin_match else None,
            "github": (github_match.group() if github_match.group().startswith("http") else "https://" + github_match.group()) if github_match else None
        }

    def extract_skills(self) -> Dict[str, List[str]]:
        """Categorizes candidate skills using word boundaries."""
        categorized_skills = {}
        for category, skills in self.SKILL_TAXONOMY.items():
            found = set()
            for skill_pattern in skills:
                pattern = r'\b' + skill_pattern + r'\b'
                if re.search(pattern, self.raw_text, re.IGNORECASE):
                    clean_name = skill_pattern.replace(r'\+', '+').replace(r'\.', '.')
                    found.add(clean_name)
            categorized_skills[category] = sorted(list(found))
        return categorized_skills

    def extract_education(self) -> List[Dict[str, str]]:
        """Parses education history and replaces unicode em-dashes with standard dashes."""
        lines = self.sections.get("EDUCATION", [])
        records = []
        current = None

        for line in lines:
            clean_line = line.replace('\u2014', '-').replace('—', '-').strip()
            lower = clean_line.lower()

            if any(p in lower for p in self.DEGREE_PATTERNS):
                if current:
                    records.append(current)
                current = {"degree": clean_line, "institution": "", "score": "", "year": ""}
            elif current:
                if any(kw in lower for kw in ["sgpa", "cgpa", "percentage", "%"]):
                    current["score"] = clean_line
                elif any(kw in lower for kw in ["graduation", "completion", "expected", "year"]):
                    current["year"] = clean_line
                elif not current["institution"]:
                    current["institution"] = clean_line

        if current:
            records.append(current)
        return records

    def extract_experience(self) -> List[Dict[str, Any]]:
        """Parses employment history."""
        lines = self.sections.get("EXPERIENCE", [])
        experiences = []
        current = None

        for line in lines:
            if "intern" in line.lower() or "developer" in line.lower():
                if current:
                    experiences.append(current)
                current = {"role_and_company": line, "description": []}
            elif current:
                if any(proj_head in line.lower() for proj_head in ["iot robotics projects", "remote controlled rover", "research project:", "smart india hackathon"]):
                    break
                current["description"].append(line)

        if current:
            experiences.append(current)
        return experiences

    def extract_projects(self) -> List[Dict[str, Any]]:
        """Parses technical & academic projects."""
        lines = self.sections.get("PROJECTS", [])
        projects = []
        current = None

        for line in lines:
            if "intern" in line.lower() and "euphoria" in line.lower():
                continue

            if any(kw in line.lower() for kw in ["rover", "workshop", "research project", "smart india hackathon"]):
                if current:
                    projects.append(current)
                if ":" in line:
                    parts = line.split(":", 1)
                    current = {"title": parts[0].strip(), "description": [parts[1].strip()] if parts[1].strip() else []}
                else:
                    current = {"title": line, "description": []}
            elif current:
                current["description"].append(line)

        if current:
            projects.append(current)
        return projects

    def extract_certifications(self) -> List[str]:
        """Parses certifications and merges multi-line bullet entries."""
        raw_lines = self.sections.get("CERTIFICATIONS", [])
        cleaned = []
        for line in raw_lines:
            # If line starts lowercase or looks like a sentence continuation, join to previous item
            if cleaned and (line[0].islower() or line.startswith("accepted")):
                cleaned[-1] = f"{cleaned[-1]} {line}"
            else:
                cleaned.append(line)
        return cleaned

    def extract_career_objective(self) -> str:
        """Joins objective section lines into a clean paragraph."""
        lines = self.sections.get("OBJECTIVE", [])
        return " ".join(lines).strip()

    def parse(self) -> Dict[str, Any]:
        """Executes full parsing pipeline and returns standardized JSON object."""
        self.extract_layout_sorted_text()
        self.extract_sections()

        return {
            "personal_information": self.extract_personal_information(),
            "career_objective": self.extract_career_objective(),
            "education": self.extract_education(),
            "skills": self.extract_skills(),
            "experience": self.extract_experience(),
            "projects": self.extract_projects(),
            "certifications": self.extract_certifications(),
            "languages": self.sections.get("LANGUAGES", [])
        }


def upload_and_parse_resume():
    """Prompts interactive file upload and parses the resume."""
    print("Please select and upload a Resume PDF file:")
    uploaded = files.upload()

    if not uploaded:
        print("No file selected.")
        return None

    uploaded_filename = list(uploaded.keys())[0]
    print(f"\n✓ File '{uploaded_filename}' uploaded successfully. Parsing...\n")

    parser = ResumeParser(uploaded_filename)
    return parser.parse()


# print("✓ Cell 2 Complete: Master ResumeParser Engine defined successfully.")
# ==========================================
# CELL 3: Execution & Output Generation
# ==========================================

# Prompts file upload and executes parsing pipeline
parsed_resume_json = upload_and_parse_resume()

# Display formatted JSON
if parsed_resume_json:
    print("==========================================")
    print("         PARSED RESUME JSON OUTPUT        ")
    print("==========================================")
    print(json.dumps(parsed_resume_json, indent=2))

Installing pdfplumber...
Installing pymupdf...
Installing scikit-learn...


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Please select and upload a Resume PDF file:


Saving Pritam Ghorai Resume Networking.pdf to Pritam Ghorai Resume Networking.pdf

✓ File 'Pritam Ghorai Resume Networking.pdf' uploaded successfully. Parsing...

         PARSED RESUME JSON OUTPUT        
{
  "personal_information": {
    "name": "Pritam Ghorai",
    "email": "pritamghorai3005@gmail.com",
    "phone": "+91 9382932972",
    "linkedin": "https://www.linkedin.com/in/pritam-ghorai-a7598031a",
    "github": "https://github.com/pritam-3005"
  },
  "career_objective": "B.Tech student in Computer Science & Engineering (AI & ML) with a strong foundation in Java, Python, C, SQL, DBMS, Data Structures andComputer Networking. Seeking a Software Trainee opportunity to apply programming and problem-solving skills in software development, testing and debugging while continuously learning and contributing to real-world applications.",
  "education": [
    {
      "degree": "\u25cf\u200b Bachelor of Technology in Computer Science & Engineering (AI & ML)",
      "institution": "Brainwa

# SKILL GAP ANALYZER

In [ ]:
# ==========================================
# CELL 1: Dataset Loader & Storage Handler
# ==========================================

import os
import pandas as pd
from google.colab import files

DATASET_FILENAME = "AI_Career_Coach_CSE_Dataset_cleand.xlsx"

def load_job_dataset(filename=DATASET_FILENAME):
    """
    Checks if the job roles dataset exists locally.
    Prompts upload ONLY if the dataset is missing.
    """
    if os.path.exists(filename):
        print(f"✓ Dataset '{filename}' found in local environment. Loading...")
    else:
        print(f"Dataset '{filename}' not found. Please upload it once:")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("Dataset file was not uploaded.")
        filename = list(uploaded.keys())[0]

    df = pd.read_excel(filename)
    print(f"✓ Successfully loaded {len(df)} job roles from dataset.")
    return df, filename

# Load dataset once
job_dataset_df, dataset_filepath = load_job_dataset()

Dataset 'AI_Career_Coach_CSE_Dataset_cleand.xlsx' not found. Please upload it once:


Saving AI_Career_Coach_CSE_Dataset_cleand.xlsx to AI_Career_Coach_CSE_Dataset_cleand.xlsx
✓ Successfully loaded 25 job roles from dataset.


In [ ]:
# ==========================================
# CELL 2: Skill Gap Analyzer & Roadmap Engine
# ==========================================

import re
import json
import pandas as pd
from typing import Dict, List, Any, Set

class SkillGapAnalyzer:
    """
    Module 2 & 3 Engine: Matches candidate skills against benchmark job roles,
    identifies minimum & advanced skill gaps, and generates tailored roadmaps.
    """

    def __init__(self, dataset_df: pd.DataFrame):
        self.df = dataset_df

    def get_available_roles(self) -> List[str]:
        """Returns sorted list of all job roles available in the dataset."""
        return sorted(self.df['Job Role'].dropna().unique().tolist())

    @staticmethod
    def parse_skill_string(skill_str: Any) -> List[str]:
        """Splits comma/slash/or separated skill strings into clean lists."""
        if not isinstance(skill_str, str) or not skill_str.strip():
            return []
        clean = skill_str.replace(' or ', ', ').replace('/', ', ')
        return [s.strip() for s in clean.split(',') if s.strip()]

    def extract_flattened_candidate_skills(self, parsed_resume_json: Dict[str, Any]) -> Set[str]:
        """Flatten candidate skills from Module 1 JSON output and expand tech stacks."""
        user_skills = set()
        if not parsed_resume_json:
            return user_skills

        # Pull categorized skills
        skills_dict = parsed_resume_json.get("skills", {})
        for cat, s_list in skills_dict.items():
            for skill in s_list:
                user_skills.add(skill.strip())

        # Tech Stack Implicit Expansions
        upper_skills = {s.upper() for s in user_skills}
        if "MERN" in upper_skills or "MERN STACK" in upper_skills:
            user_skills.update([
                "MongoDB", "Express", "React", "Node.js", "HTML",
                "CSS", "JavaScript", "REST API", "REST APIs", "Web Development"
            ])

        return user_skills

    @staticmethod
    def is_skill_matched(target_skill: str, user_skills: Set[str]) -> bool:
        """Normalized string boundary comparison for skill matching."""
        target_norm = re.sub(r'[^a-zA-Z0-9]', '', target_skill.lower())
        for u in user_skills:
            u_norm = re.sub(r'[^a-zA-Z0-9]', '', u.lower())
            if target_norm == u_norm or target_norm in u_norm or u_norm in target_norm:
                return True
        return False

    def analyze_gap(self, selected_role: str, parsed_resume_json: Dict[str, Any]) -> Dict[str, Any]:
        """Executes skill gap analysis for candidate against selected job role."""
        role_rows = self.df[self.df['Job Role'] == selected_role]
        if role_rows.empty:
            raise ValueError(f"Job Role '{selected_role}' not found in dataset.")

        row = role_rows.iloc[0]
        min_skills = self.parse_skill_string(row['Minimum Required Skills'])
        adv_skills = self.parse_skill_string(row['Preferred / Advanced Skills'])
        certifications = str(row['Recommended Certifications'])
        category = str(row['Category'])

        user_skills = self.extract_flattened_candidate_skills(parsed_resume_json)

        matched_min = [s for s in min_skills if self.is_skill_matched(s, user_skills)]
        missing_min = [s for s in min_skills if not self.is_skill_matched(s, user_skills)]

        matched_adv = [s for s in adv_skills if self.is_skill_matched(s, user_skills)]
        missing_adv = [s for s in adv_skills if not self.is_skill_matched(s, user_skills)]

        total_min = len(min_skills)
        match_score = round((len(matched_min) / total_min) * 100) if total_min > 0 else 100

        return {
            "job_role": selected_role,
            "category": category,
            "match_score": match_score,
            "min_skills_required": min_skills,
            "matched_min_skills": matched_min,
            "missing_min_skills": missing_min,
            "adv_skills_required": adv_skills,
            "matched_adv_skills": matched_adv,
            "missing_adv_skills": missing_adv,
            "recommended_certifications": certifications
        }

    def generate_career_roadmap(self, gap_results: Dict[str, Any]) -> str:
        """Generates a structured, phase-by-phase learning roadmap."""
        role = gap_results["job_role"]
        missing_min = gap_results["missing_min_skills"]
        missing_adv = gap_results["missing_adv_skills"]
        certs = gap_results["recommended_certifications"]

        lines = []
        lines.append(f"### 🚀 Actionable Learning Roadmap: {role}\n")

        if not missing_min and not missing_adv:
            lines.append("🎉 **Congratulations! You already possess all minimum and advanced skills for this role.**\n")
            lines.append("👉 **Next Steps:** Focus on building 1-2 production-grade capstone projects and pursuing recommended certifications to stand out to recruiters.\n")
        else:
            phase = 1
            if missing_min:
                lines.append(f"#### **Phase {phase}: Core Skill Acquisition (Weeks 1 - 4)**")
                lines.append("Focus on mastering the mandatory minimum skills required for entry-level competence:\n")
                for skill in missing_min:
                    lines.append(f"- **{skill}**: Master core syntax, fundamentals, and complete 2-3 focused coding drills.")
                lines.append("")
                phase += 1

            if missing_adv:
                lines.append(f"#### **Phase {phase}: Advanced & Industry Standard Tools (Weeks 5 - 8)**")
                lines.append("Strengthen your profile by learning preferred advanced skills:\n")
                for skill in missing_adv:
                    lines.append(f"- **{skill}**: Study architecture principles, integrate into real-world modules, and explore best practices.")
                lines.append("")
                phase += 1

            lines.append(f"#### **Phase {phase}: Portfolio Capstone & Certification (Weeks 9 - 12)**")
            all_gaps = missing_min + missing_adv
            skills_str = ", ".join(all_gaps) if all_gaps else "your target domain skills"
            lines.append(f"- **Capstone Project**: Build an end-to-end full project incorporating: *{skills_str}*.")
            lines.append(f"- **Certification Prep**: Prepare for recommended industry credentials: **{certs}**.")
            lines.append("- **Resume & GitHub Update**: Publish full source code with clean documentation on GitHub and update your resume skills section.")

        return "\n".join(lines)

# Initialize Analyzer
gap_engine = SkillGapAnalyzer(job_dataset_df)

In [ ]:
# ==========================================
# CELL 3: Interactive Dashboard UI (Dropdown Placeholder Update)
# ==========================================

import ipywidgets as widgets
from IPython.display import display, Markdown

# Fallback check for parsed_resume_json
if 'parsed_resume_json' not in globals() or parsed_resume_json is None:
    parsed_resume_json = {
        "skills": {
            "programming_languages": ["C", "Java", "JavaScript", "Python"],
            "frameworks_and_libraries": ["MERN"],
            "databases": [],
            "tools_and_cloud": ["Arduino", "GitHub", "Microcontroller"],
            "soft_skills": ["Communication", "Problem Solving", "Teamwork"]
        }
    }

# Build role options with "Select Job Role" as default placeholder
available_roles = ["Select Job Role"] + gap_engine.get_available_roles()

# Widget Controls
role_dropdown = widgets.Dropdown(
    options=available_roles,
    value="Select Job Role",
    description='Target Role:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

analyze_button = widgets.Button(
    description='Analyze Skill Gap & Roadmap',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='30%', margin='10px 0px')
)

output_area = widgets.Output()


def on_button_clicked(b):
    with output_area:
        output_area.clear_output()
        selected_role = role_dropdown.value

        # Check if user selected the default placeholder
        if selected_role == "Select Job Role":
            display(Markdown("⚠️ **Please select a valid Job Role from the dropdown menu first.**"))
            return

        # Execute Skill Gap Analysis
        res = gap_engine.analyze_gap(selected_role, parsed_resume_json)
        roadmap_md = gap_engine.generate_career_roadmap(res)

        # Build Clean Markdown Text Output
        matched_min_str = ", ".join([f"`{s}`" for s in res['matched_min_skills']]) if res['matched_min_skills'] else "None matched yet"
        missing_min_str = ", ".join([f"`{s}`" for s in res['missing_min_skills']]) if res['missing_min_skills'] else "None! All minimum skills present."
        missing_adv_str = ", ".join([f"`{s}`" for s in res['missing_adv_skills']]) if res['missing_adv_skills'] else "None! All advanced skills present."

        output_md = f"""
## 🎯 Skill Gap Analysis: {res['job_role']}

**Category:** {res['category']}
**Role Qualification Score:** {res['match_score']}%

---

* **✅ Skills You Already Possess (Matched Required):**
  {matched_min_str}

* **❌ Critical Skill Gaps (Minimum Skills to Learn):**
  {missing_min_str}

* **⚡ Preferred / Advanced Skills to Stand Out:**
  {missing_adv_str}

* **📜 Recommended Industry Certifications:**
  {res['recommended_certifications']}

---

{roadmap_md}
        """

        display(Markdown(output_md))


analyze_button.on_click(on_button_clicked)

# Render Controls
display(Markdown("### 🛠️ AI Career Coach — Select Target Job Role"))
display(role_dropdown)
display(analyze_button)
display(output_area)

### 🛠️ AI Career Coach — Select Target Job Role

Dropdown(description='Target Role:', layout=Layout(width='50%'), options=('Select Job Role', 'AI Engineer', 'A…

Button(button_style='primary', description='Analyze Skill Gap & Roadmap', icon='search', layout=Layout(margin=…

Output()

# PLACEMENT PREDICTOR

In [ ]:
# ==========================================
# CELL 3: Placement Predictor & Best Role Finder UI
# ==========================================

import re
import ipywidgets as widgets
from IPython.display import display, Markdown

# Fallback check for parsed_resume_json
if 'parsed_resume_json' not in globals() or parsed_resume_json is None:
    parsed_resume_json = {
        "skills": {
            "programming_languages": ["C", "Java", "JavaScript", "Python"],
            "frameworks_and_libraries": ["MERN"],
            "databases": [],
            "tools_and_cloud": ["Arduino", "GitHub", "Microcontroller"],
            "soft_skills": ["Communication", "Problem Solving", "Teamwork", "Leadership"]
        },
        "experience": [{"role_and_company": "MERN Stack Developer Intern"}],
        "projects": [{"title": "IoT Smart Container"}]
    }


class PlacementPredictorEngine:
    """Calculates placement probabilities and identifies optimal candidate-role fit."""

    def __init__(self, gap_analyzer):
        self.analyzer = gap_analyzer

    def predict_placement_probability(self, role: str, resume_json: dict) -> dict:
        """Calculates weighted placement probability for a target role."""
        gap_data = self.analyzer.analyze_gap(role, resume_json)

        min_skills = gap_data["min_skills_required"]
        matched_min = gap_data["matched_min_skills"]
        adv_skills = gap_data["adv_skills_required"]
        matched_adv = gap_data["matched_adv_skills"]

        # 1. Minimum Skills Weight (70%)
        min_score = (len(matched_min) / len(min_skills)) * 70 if min_skills else 70

        # 2. Advanced Skills Weight (20%)
        adv_score = (len(matched_adv) / len(adv_skills)) * 20 if adv_skills else 20

        # 3. Practical Experience Boost (10%)
        has_internship = len(resume_json.get("experience", [])) > 0
        has_projects = len(resume_json.get("projects", [])) > 0
        profile_boost = (5 if has_internship else 0) + (5 if has_projects else 0)

        total_probability = min(round(min_score + adv_score + profile_boost), 98)

        gap_data["placement_probability"] = total_probability
        return gap_data

    def find_best_matching_role(self, resume_json: dict) -> dict:
        """Evaluates all roles in dataset and returns the role with the highest probability."""
        all_roles = self.analyzer.get_available_roles()
        predictions = []

        for role in all_roles:
            pred = self.predict_placement_probability(role, resume_json)
            predictions.append(pred)

        # Sort descending by probability
        predictions.sort(key=lambda x: x["placement_probability"], reverse=True)
        return predictions[0]


# Initialize Predictor
predictor_engine = PlacementPredictorEngine(gap_engine)

# Dropdown Menu Controls
available_roles = ["Select Job Role"] + gap_engine.get_available_roles()

role_dropdown = widgets.Dropdown(
    options=available_roles,
    value="Select Job Role",
    description='Target Role:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

analyze_button = widgets.Button(
    description='Analyze Placement & Skill Gap',
    button_style='primary',
    icon='chart-line',
    layout=widgets.Layout(width='35%', margin='10px 0px')
)

output_area = widgets.Output()


def on_button_clicked(b):
    with output_area:
        output_area.clear_output()
        selected_role = role_dropdown.value

        if selected_role == "Select Job Role":
            display(Markdown("⚠️ **Please select a valid Job Role from the dropdown menu first.**"))
            return

        # 1. Evaluate Selected Role Placement Probability
        target_res = predictor_engine.predict_placement_probability(selected_role, parsed_resume_json)
        target_roadmap = gap_engine.generate_career_roadmap(target_res)

        # 2. Automatically Find Best Role for Candidate Profile
        best_role_res = predictor_engine.find_best_matching_role(parsed_resume_json)

        # Candidate Skills Formatting for Best Role
        best_has_str = ", ".join([f"`{s}`" for s in best_role_res['matched_min_skills'] + best_role_res['matched_adv_skills']]) or "None"
        best_req_str = ", ".join([f"`{s}`" for s in best_role_res['min_skills_required']]) or "None"
        best_missing_str = ", ".join([f"`{s}`" for s in best_role_res['missing_min_skills']]) if best_role_res['missing_min_skills'] else "None! Profile is fully qualified."

        # Selected Role Formatting
        target_has_str = ", ".join([f"`{s}`" for s in target_res['matched_min_skills']]) if target_res['matched_min_skills'] else "None matched yet"
        target_missing_str = ", ".join([f"`{s}`" for s in target_res['missing_min_skills']]) if target_res['missing_min_skills'] else "None! All minimum skills present."
        target_adv_str = ", ".join([f"`{s}`" for s in target_res['missing_adv_skills']]) if target_res['missing_adv_skills'] else "None! All advanced skills present."

        output_md = f"""
## 🎯 Placement Prediction & Skill Gap Analysis

### 📊 Selected Target Role: **{target_res['job_role']}**
* **Category:** {target_res['category']}
* **Placement Probability:** **{target_res['placement_probability']}%**
* **Qualification Score:** {target_res['match_score']}%

---

### 🌟 AI Recommended Best Fit Role: **{best_role_res['job_role']}**
Based on your parsed resume skills, internships, and projects, this role offers your **highest placement probability**.

* **Best Role Placement Probability:** **{best_role_res['placement_probability']}%**
* **Category:** {best_role_res['category']}
* **Minimum Required Skills:** {best_req_str}
* **Skills You Already Have for this Role:** {best_has_str}
* **Remaining Skill Gaps for Best Role:** {best_missing_str}

---

### 🛠️ Detailed Analysis for Selected Role ({target_res['job_role']})

* **✅ Skills You Already Possess (Matched Required):**
  {target_has_str}

* **❌ Critical Skill Gaps (Minimum Skills to Learn):**
  {target_missing_str}

* **⚡ Preferred / Advanced Skills to Stand Out:**
  {target_adv_str}

* **📜 Recommended Industry Certifications:**
  {target_res['recommended_certifications']}

---

{target_roadmap}
        """

        display(Markdown(output_md))


analyze_button.on_click(on_button_clicked)

# Render Controls
display(Markdown("### 🛠️ AI Career Coach — Select Target Job Role"))
display(role_dropdown)
display(analyze_button)
display(output_area)

### 🛠️ AI Career Coach — Select Target Job Role

Dropdown(description='Target Role:', layout=Layout(width='50%'), options=('Select Job Role', 'AI Engineer', 'A…

Button(button_style='primary', description='Analyze Placement & Skill Gap', icon='chart-line', layout=Layout(m…

Output()